## Install Dependencies

In [ ]:
!pip install langchain langchain-groq langchain-huggingface langchain-chroma sentence-transformers

## Imports

In [ ]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

## Setup Components

In [ ]:
os.environ['GROQ_API_KEY'] = 'gsk_YOUR_API_KEY_HERE'

# Create dummy data
with open("dummy_data.txt", "w") as f:
    f.write("The company refund policy allows returns within 30 days. Software purchases are non-refundable.")

chunks = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20).split_documents(TextLoader("dummy_data.txt").load())

vectorstore = Chroma.from_documents(chunks, HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

llm = ChatGroq(model='llama-3.3-70b-versatile')

template = """Answer based ONLY on context:\n{context}\n\nQuestion: {question}"""
prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

## RAG Pipeline Function

In [ ]:
def query_pdf(query):
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return rag_chain.invoke(query)

## Test Basic RAG

In [ ]:
answer = query_pdf("What is the refund policy for software?")